# Comparing Inner Transport Solvers

![Reflecting scattering slab used to compare the inner solvers](images/solver_comparison_slab.png)

This tutorial solves the same scattering problem with all four inner transport methods available in OpenSn, then compares their converged scalar fluxes, transport-sweep counts, and wall times.

## Solve a common reference problem

The available choices are OpenSn's explicit classic Richardson iteration and PETSc's Richardson, GMRES, and BiCGStab methods. The physical model, convergence tolerance, and iteration limit are unchanged between runs. A homogeneous reflecting slab provides an analytic energy-summed, volume-averaged scalar flux of $1/\Sigma_a=5$. A 1000-cell mesh and 32-direction quadrature make the timing comparison measurable while keeping the problem inexpensive. The timings remain illustrative rather than a general performance ranking.

In [ ]:
from mpi4py import MPI
from pyopensn.aquad import GLProductQuadrature1DSlab
from pyopensn.context import Finalize
from pyopensn.mesh import OrthogonalMeshGenerator
from pyopensn.post import VolumePostprocessor
from pyopensn.solver import DiscreteOrdinatesProblem, SteadyStateSourceSolver
from pyopensn.source import VolumetricSource
from pyopensn.xs import MultiGroupXS

comm = MPI.COMM_WORLD
rank = comm.rank
methods = {
    "classic_richardson": "Classic Richardson",
    "petsc_richardson": "PETSc Richardson",
    "petsc_gmres": "PETSc GMRES",
    "petsc_bicgstab": "PETSc BiCGStab",
}


def solve_with(method):
    mesh = OrthogonalMeshGenerator(node_sets=[[i / 1000.0 for i in range(1001)]]).Execute()
    mesh.SetUniformBlockID(0)
    xs = MultiGroupXS()
    xs.CreateSimpleOneGroup(sigma_t=1.0, c=0.8)
    source = VolumetricSource(block_ids=[0], group_strength=[1.0])
    quadrature = GLProductQuadrature1DSlab(n_polar=32, scattering_order=0)
    groupset = {
        "groups_from_to": (0, 0),
        "angular_quadrature": quadrature,
        "inner_linear_method": method,
        "l_abs_tol": 1.0e-8,
        "l_max_its": 500,
    }
    if method == "petsc_gmres":
        groupset["gmres_restart_interval"] = 30

    problem = DiscreteOrdinatesProblem(
        mesh=mesh,
        num_groups=1,
        groupsets=[groupset],
        xs_map=[{"block_ids": [0], "xs": xs}],
        volumetric_sources=[source],
        boundary_conditions=[
            {"name": "zmin", "type": "reflecting"},
            {"name": "zmax", "type": "reflecting"},
        ],
        options={"verbose_inner_iterations": False},
    )
    solver = SteadyStateSourceSolver(problem=problem)
    comm.Barrier()
    start = MPI.Wtime()
    solver.Initialize()
    solver.Execute()
    comm.Barrier()
    elapsed = comm.allreduce(MPI.Wtime() - start, op=MPI.MAX)
    average = VolumePostprocessor(problem=problem, value_type="avg")
    average.Execute()
    total_average_flux = sum(average.GetValue()[0])
    return total_average_flux, solver.GetNumSweeps(), elapsed

results = {method: solve_with(method) for method in methods}

## Compare the converged solutions

Solver choice should affect the convergence path, not the converged physical solution. Classic Richardson is used as the flux-comparison baseline. Sweep counts are a machine-independent measure of transport work; wall times include solver initialization and execution and will vary by machine. Inner iteration logging is disabled so console output does not distort these short timings. Set `verbose_inner_iterations=True` to inspect the histories, remembering that the classic method uses a pointwise flux-change test while the PETSc methods use a scaled residual.

In [ ]:
reference_flux = results["classic_richardson"][0]
flux_differences = {
    method: abs(flux - reference_flux)
    for method, (flux, _, _) in results.items()
}
maximum_flux_difference = max(flux_differences.values())
maximum_analytic_error = max(
    abs(flux - 5.0) for flux, _, _ in results.values()
)

if rank == 0:
    for method, label in methods.items():
        flux, sweeps, elapsed = results[method]
        print(f"{label} total average flux={flux:.12e}")
        print(f"{label} flux difference={flux_differences[method]:.12e}")
        print(f"{label} sweeps={sweeps}")
        print(f"{label} wall time (s)={elapsed:.6f}")
    print(f"Maximum inner-solver flux difference={maximum_flux_difference:.12e}")

assert maximum_flux_difference < 1.0e-5
assert maximum_analytic_error < 1.0e-5

A representative one-process run gives:

| Inner method | Total average flux | Difference from classic Richardson | Transport sweeps | Wall time (s) |
|---|---:|---:|---:|---:|
| Classic Richardson | 4.999999963924 | 0 | 107 | 0.5685 |
| PETSc Richardson | 4.999999963924 | $5.24\times10^{-14}$ | 108 | 0.5473 |
| PETSc GMRES | 4.999999988698 | $2.48\times10^{-8}$ | 10 | 0.0326 |
| PETSc BiCGStab | 4.999999999905 | $3.60\times10^{-8}$ | 14 | 0.0837 |

All four methods reproduce the analytic flux within $4\times10^{-8}$ and agree with classic Richardson within $4\times10^{-8}$. For this problem, PETSc Richardson performs nearly the same sweep work as classic Richardson, while GMRES and BiCGStab reduce the sweep count by approximately 91% and 87%, respectively. Wall time follows the reduction in sweep work in this representative run, but exact timings depend on the machine and should not be treated as a universal solver ranking.

## Finalize (for Jupyter Notebook only)

In script mode, PyOpenSn handles finalization automatically. In a Jupyter kernel, finalize OpenSn before MPI.

In [ ]:
if "opensn_console" not in globals():
    from IPython import get_ipython

    if get_ipython() is not None:
        Finalize()
        MPI.Finalize()